# Data reader

Read the processed SciX papers and make same-author (`1`) and different-author (`0`) pairs. Only titles and abstracts go into the text; ORCIDs give us labels, and bibcodes track the papers.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import pyarrow.parquet as parquet

ROOT = Path.cwd().resolve()
if ROOT.name in ('2_Data', '3_train'):
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data/processed/sciar_refereed_v0/papers.parquet'
COLUMNS = ['bibcode', 'first_author_orcid', 'title', 'abstract']

In [2]:
def load_papers(path=DATA_PATH, max_papers=None):
    # only read the columns we need; the demo reads one small batch
    if max_papers is None:
        papers = pd.read_parquet(path, columns=COLUMNS)
    else:
        batch = next(parquet.ParquetFile(path).iter_batches(batch_size=max_papers, columns=COLUMNS))
        papers = batch.to_pandas()

    # only keeping papers with the fields needed for pairs
    papers = papers.dropna(subset=COLUMNS)
    for column in COLUMNS:
        papers = papers[papers[column].str.strip().ne('')]
    return papers.reset_index(drop=True)


def split_authors(papers, seed=42):
    # split authors first so we don't leak authors between sets
    authors = np.array(sorted(papers['first_author_orcid'].unique()))
    np.random.default_rng(seed).shuffle(authors)
    train_end = int(len(authors) * 0.8)
    val_end = train_end + int(len(authors) * 0.1)
    return {
        'train': authors[:train_end].tolist(),
        'validation': authors[train_end:val_end].tolist(),
        'test': authors[val_end:].tolist(),
    }

In [3]:
def make_pairs(papers, paper_groups, author_ids, n_pairs, seed=42):
    if n_pairs < 2 or n_pairs % 2:
        raise ValueError('n_pairs must be a positive even number')

    # single-paper authors can still show up in different-author pairs
    positive_authors = [author for author in author_ids if len(paper_groups[author]) >= 2]
    if not positive_authors or len(author_ids) < 2:
        raise ValueError('Need a two-paper author and two different authors in each split')

    rng = random.Random(seed)
    pairs = []
    for _ in range(n_pairs // 2):
        # same author pairs get a 1; use two different papers
        author = rng.choice(positive_authors)
        left, right = rng.sample(list(paper_groups[author]), 2)
        pairs.append((left, right, 1))

        # grab a random paper from two different authors
        author_a, author_b = rng.sample(author_ids, 2)
        left = rng.choice(paper_groups[author_a])
        right = rng.choice(paper_groups[author_b])
        pairs.append((left, right, 0))

    rng.shuffle(pairs)
    examples = []
    for left, right, label in pairs:
        a, b = papers.iloc[left], papers.iloc[right]
        examples.append({
            'text_a': a['title'] + '\n\n' + a['abstract'],
            'text_b': b['title'] + '\n\n' + b['abstract'],
            'label': label,
            'bibcode_a': a['bibcode'],
            'bibcode_b': b['bibcode'],
        })
    return pd.DataFrame(examples)

In [4]:
# using a small sample and a fixed seed so I can reproduce this
papers = load_papers(max_papers=10_000)
author_splits = split_authors(papers, seed=42)
paper_groups = papers.groupby('first_author_orcid').indices
pair_sets = {
    name: make_pairs(papers, paper_groups, ids, n_pairs=10, seed=42 + offset)
    for offset, (name, ids) in enumerate(author_splits.items())
}

In [5]:
# checking that we didn't accidentally reuse an author or mislabel a pair
train, val, test = (set(ids) for ids in author_splits.values())
assert not (train & val or train & test or val & test)
by_bibcode = papers.set_index('bibcode')['first_author_orcid']
for name, examples in pair_sets.items():
    for row in examples.itertuples():
        assert (by_bibcode[row.bibcode_a] == by_bibcode[row.bibcode_b]) == (row.label == 1)
        assert by_bibcode[row.bibcode_a] in author_splits[name]
        assert by_bibcode[row.bibcode_b] in author_splits[name]
        if row.label == 1:
            assert row.bibcode_a != row.bibcode_b
    print(f'{name}: {len(author_splits[name])} authors, {len(examples)} pairs')

print('No authors overlap between splits. Labels checked.')
for row in pair_sets['train'].head(3).itertuples():
    print(f'label={row.label} | {row.bibcode_a} / {row.bibcode_b}')
    print(f'  A: {row.text_a[:100]!r}')
    print(f'  B: {row.text_b[:100]!r}')

train: 4758 authors, 10 pairs
validation: 594 authors, 10 pairs
test: 596 authors, 10 pairs
No authors overlap between splits. Labels checked.
label=1 | 2021NatSR..1114325T / 2023PhRvB.107p5429T
  A: 'Designing magnetic field sensor based on tapered photonic crystal fibre assisted by a ferrofluid\n\nA '
  B: 'Asymmetric Fraunhofer diffraction in a quantum dot molecule via tunneling-induced transparency\n\nWe m'
label=0 | 2020ElecJ..3306824C / 2022Symm...14..209R
  A: 'Prepaid electricity and in-home displays: An alternative for the most vulnerable households in Colom'
  B: 'Double Diffusive Magneto-Free-Convection Flow of Oldroyd-B Fluid over a Vertical Plate with Heat and'
label=0 | 2022OptL...47.5925G / 2019BuEGE..78.2315W
  A: 'Single-ring hollow-core anti-resonant fiber with a record low loss (4.3 dB/km) for high-power laser '
  B: 'Peak shear strength prediction for discontinuities between two different rock types using a neural n'
